# 6. 전망

자료가 끝난 뒤를 모델로 풉니다. 외생변수의 앞날 가정을 세우는 법, 가정이 비었을 때의 진단, 전망 출발점을 실측에 맞추는 법,
불확실성 띠, 그리고 지난 기간으로 모델의 예측력을 평가하는 법을 배웁니다.

In [ ]:
import equion as eqn

wf = eqn.datasets.load("toy_q")
eq_i = wf.equation("eq_i").ls("I c Y(-1) R")
eq_cn = wf.equation("eq_cn").tsls("CN c Y CN(-1) @inst G R CN(-1) Y(-1)")
eq_p = wf.equation("eq_p").ls("dlog(P) c dlog(Y) dlog(POIL)")
m = wf.model("toy").merge(eq_i, eq_cn, eq_p).append("Y = CN + I + G")
print(m.exogenous)                    # 앞날 경로가 필요한 외생변수

## 6.1 외생 가정이 없으면

자료는 2019Q4에서 끝납니다. 외생변수의 앞날을 주지 않고 전망하면, 무엇이 언제 비었는지 알려 주고 멈춥니다.

In [ ]:
try:  # 이 예는 일부러 오류를 보인다
    m.forecast("2020Q1", "2021Q4")
except Exception as e:
    print(f"{type(e).__name__}: {e}")

## 6.2 외생 가정 세우기

세 가지 방법이 있습니다. ① `exog_fill`로 규칙에 따라 한꺼번에(마지막 값 유지 `last` · 평균 성장률 `growth` · 평균 증분 `trend`),
② 워크파일 시리즈를 직접 채우기, ③ 시나리오의 `override`로 가정만 따로 두기(실측 시리즈는 그대로).

In [ ]:
r = m.forecast("2020Q1", "2021Q4", exog_fill="last")     # 기간을 늘리고 외생을 마지막 값으로 채워 푼다
print(r)
print(wf.to_pandas(["G", "R", "POIL"], labels=True).loc["2019Q4":"2020Q2"].round(2))

In [ ]:
alt = m.scenario("plan")                                  # 다른 가정: 정부지출 연 4% 증가, 금리 인하
g0 = wf.elem("G", "2019Q4")
alt.override("G", {f"{y}Q{q}": g0 * 1.01 ** (4 * (y - 2020) + q) for y in (2020, 2021) for q in (1, 2, 3, 4)})
alt.override("R", {f"2020Q{q}": 2.5 for q in (1, 2, 3, 4)} | {f"2021Q{q}": 2.0 for q in (1, 2, 3, 4)})
m.forecast("2020Q1", "2021Q4", scenario="plan")
print(wf.to_pandas(["Y_0", "Y" + alt.alias], labels=True).loc["2020Q1":"2021Q4"].round(1))

`exog_fill`은 비어 있는 외생변수만 채웁니다(이미 값이 있으면 그대로). 시나리오의 `override`는 워크파일의 실측 시리즈를 바꾸지
않으므로, 여러 가정을 나란히 두기 좋습니다.

## 6.3 출발점 맞추기

모델은 마지막 실측 분기(2019Q4)에서도 잔차만큼 실측과 다릅니다. 그 잔차를 앞으로도 이어 붙이면(`init_add_factor`) 전망이
실측에서 매끄럽게 출발합니다 — 실무에서 흔히 쓰는 '착지'입니다.

In [ ]:
land = m.scenario("landed")
for eq in ("eq_i", "eq_cn", "eq_p"):
    land.init_add_factor(eq, "2019Q4")
m.forecast("2020Q1", "2021Q4", scenario="landed")
print(wf.to_pandas(["CN", "CN_0", "CN" + land.alias], labels=True).loc["2019Q4":"2020Q2"].round(2))

## 6.4 불확실성 띠

In [ ]:
m.scenario("baseline")
sto = m.stochastic("2020Q1", "2021Q4", reps=300, seed=7)
b = sto.band("Y")
b.index = [f"{d.year}Q{(d.month - 1) // 3 + 1}" for d in b.index]
print(b.round(1))

`p5`~`p95`가 90% 띠입니다. 시차가 쌓일수록 띠가 넓어집니다. `coef_uncertainty=True`면 계수 추정의 불확실성도 더합니다.

## 6.5 예측력 평가 — 지난 기간으로

모델의 예측력은 추정에 쓰지 않은 기간을 맞춰 보며 잽니다. 2009년까지로 추정하고 2010~2019년을 풀어 실측과 견줍니다.
비교 대상으로 '전 분기와 같다'(무변화 예측)를 둡니다.

In [ ]:
ev = eqn.datasets.load("toy_q")
ev.smpl("1970Q1", "2009Q4")                               # 추정 구간
for name, spec in (("i", "I c Y(-1) R"), ("p", "dlog(P) c dlog(Y) dlog(POIL)")):
    ev.equation("e_" + name).ls(spec)
ev.equation("e_cn").tsls("CN c Y CN(-1) @inst G R CN(-1) Y(-1)")
mv = ev.model("mv").merge("e_i", "e_cn", "e_p").append("Y = CN + I + G")
mv.solve("2010Q1", "2019Q4", mode="static")               # 한 분기 앞 예측(시차는 실측)
ev.smpl("2010Q1", "2019Q4")                               # 평가 구간 — genr도 표본 안만 쓰므로 먼저 바꾼다
ev.genr("Y_NAIVE = Y(-1)")
eqn.show(ev.forecast_eval("Y", "Y_0", "Y_NAIVE"))
ev.dm_test("Y", "Y_0", "Y_NAIVE")                          # 두 예측의 정확도가 같은가(Diebold-Mariano)

`genr`은 표본 안만 계산하므로, 무변화 예측 `Y_NAIVE`를 만들기 전에 평가 구간으로 표본을 바꿨습니다.
RMSE·MAE·MAPE가 작을수록, Theil U가 0에 가까울수록 좋습니다. Diebold-Mariano 통계량이 음수이고 p값이 작으면 첫 예측(모델)이
유의하게 더 정확하다는 뜻입니다. 동적 풀이(`mode="dynamic"`)로 하면 여러 분기 앞 예측의 정확도가 됩니다.

## 6.6 결과 내보내기

In [ ]:
out = wf.to_pandas(["Y_0", "CN_0", "I_0", "P_0"], labels=True).loc["2020Q1":"2021Q4"]
out.to_excel("forecast.xlsx")                              # pandas 그대로
wf.write("forecast.csv", ["Y_0", "Y_1"])                    # 또는 워크파일에서
print(out.round(1).head(2))

## 정리

| 하려는 일 | 코드 |
|---|---|
| 전망 | `m.forecast(start, end, exog_fill="last"\|"growth"\|"trend", scenario=…)` |
| 가정 | `wf.fill("G", method=…)` · `wf["G"] = …` · `sc.override("G", {기간: 값})` |
| 착지 | `sc.init_add_factor("eq_cn", "2019Q4")` |
| 띠 | `m.stochastic(start, end, reps=300).band("Y")` |
| 평가 | `wf.forecast_eval("Y", "Y_0", "Y_NAIVE")` · `wf.dm_test("Y", "Y_0", "Y_NAIVE")` |

다음: [7. 시스템·VAR·VECM](../../docs/tutorial/07-systems.md). 레시피: [전망 레시피](../../docs/howto/forecast.md).